# 02. Feature Engineering Engine
## Transforming Raw DNS Queries into Discriminative ML Vectors

This notebook demonstrates the feature engineering pipeline implemented in `src/features.py`:
1. **Information Theory**: Shannon Entropy (full domain, subdomain, individual labels).
2. **Lexical & Structural Metrics**: Domain lengths, label counts, min/max/avg label lengths.
3. **Linguistic Ratios**: Vowel ratio, digit ratio, consonant ratio, vowel-to-consonant, hex ratio.
4. **Temporal Frequency**: Rolling query frequency (queries per minute/second) for burst vs slow attacks.
5. **Batch Processing**: Transforming raw PCAP captures or labeled CSVs into feature matrices ready for ML.

In [ ]:
import os
import sys
import yaml
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

# Ensure repo root is on sys.path
for p in [".", "..", "../.."]:
    if os.path.exists(os.path.join(p, "configs", "config.yaml")):
        sys.path.insert(0, os.path.abspath(p))
        REPO_ROOT = os.path.abspath(p)
        break

# Load configuration
config_path = os.path.join(REPO_ROOT, "configs", "config.yaml")
with open(config_path, "r") as f:
    config = yaml.safe_load(f)

from src.features import (
    shannon_entropy,
    compute_entropy,
    vowel_digit_ratio,
    max_consecutive_consonants,
    split_domain_components,
    extract_domain_features,
    calculate_query_frequency,
    process_traffic_dataset
)

print("Feature engineering modules loaded successfully.")

### 1. Information-Theoretic Metrics: Shannon Entropy
Shannon entropy quantifies the uncertainty or randomness of characters in a string:

$$H(X) = - \sum_{i=1}^n P(x_i) \log_2 P(x_i)$$

Natural domain names (e.g. `google.com`, `wikipedia.org`) have repetitive linguistic patterns resulting in low entropy.
Base32 and Hex encoded DNS tunnels (e.g. `54657374...tunnel.org`) exhibit near-maximal entropy for their character sets.

In [ ]:
# Comparing entropy across benign, Base32, and Hex queries
sample_queries = {
    "Benign - Standard": "google.com",
    "Benign - Subdomain": "cdn.assets.cloudflare.net",
    "Exfil - Base32": "sq0001.krsxg5bamnxw4ztjmrsw45djmfwcazdb.tunnel.example.com",
    "Exfil - Hex": "sq0001.5465737420636f6e666964656e7469616c.tunnel.example.com"
}

for desc, q in sample_queries.items():
    ent = compute_entropy(q)
    parts = split_domain_components(q)
    sub_ent = compute_entropy(parts["subdomain"]) if parts["subdomain"] else 0.0
    print(f"{desc:20} | FQDN: {q[:45]:45}... | Total H: {ent:.4f} | Subdomain H: {sub_ent:.4f}")

### 2. Lexical & Structural Metrics: Label Lengths & Components
According to RFC 1035:
- Labels must not exceed 63 characters.
- FQDN must not exceed 253 characters.

Attackers maximize each label to fit up to 63 bytes of payload, leading to unusually high `max_label_len` and `subdomain_len`.

In [ ]:
# Extracting component and length features
demo_features = []
for desc, q in sample_queries.items():
    feats = extract_domain_features(q)
    feats["description"] = desc
    demo_features.append(feats)

df_demo = pd.DataFrame(demo_features)
cols = ["description", "domain_len", "subdomain_len", "num_labels", "max_label_len", "avg_label_len", "entropy"]
df_demo[cols]

### 3. Linguistic Ratios: Vowel/Digit Ratios & Consonants
Normal English / human words have regular vowel-to-consonant ratios (~0.4 - 0.7).
Base32 has few vowels (`A, E, I`) relative to consonants, and Hex contains only `A-F` with `0-9` digits.

In [ ]:
# Inspecting ratio metrics
ratio_cols = ["description", "digit_ratio", "hex_ratio", "vowel_ratio", "consonant_ratio", "max_consecutive_consonants"]
df_demo[ratio_cols]

### 4. Temporal Frequency Metrics: Burst vs Low-and-Slow
DNS exfiltration attacks follow distinct operational profiles:
- **Burst mode**: Hundreds of queries sent in milliseconds/seconds to exfiltrate data quickly.
- **Low-and-Slow mode**: Covert queries sent at intervals (every 10-60s) to evade volume-based alarms.

In [ ]:
# Demonstrating rolling query frequency computation
time_records = pd.DataFrame({
    "qname": ["tunnel.exfil.org"] * 5 + ["google.com"] * 2,
    "timestamp": [1710000000.0, 1710000001.0, 1710000002.0, 1710000002.5, 1710000003.0, 1710000000.0, 1710000030.0]
})

freq_result = calculate_query_frequency(time_records, window="10s", rate_unit="second")
freq_result

### 5. Full Pipeline: Processing Raw Traffic into ML Feature Dataset

In [ ]:
# Process existing labeled traffic dataset or synthetic session
raw_traffic_path = os.path.join(REPO_ROOT, config["paths"].get("labeled_traffic_csv", "data/processed/labeled_traffic.csv"))
output_features_path = os.path.join(REPO_ROOT, config["paths"].get("features_dataset_csv", "data/processed/dns_features.csv"))

if os.path.exists(raw_traffic_path):
    print(f"Loading raw traffic from {raw_traffic_path}...")
    df_raw = pd.read_csv(raw_traffic_path)
else:
    print("Synthesizing raw traffic session...")
    from src.capture_traffic import generate_synthesized_traffic
    records, _ = generate_synthesized_traffic(benign_count=200, burst_count=120, low_slow_count=40)
    df_raw = pd.DataFrame(records)

# Execute feature extraction pipeline
df_features = process_traffic_dataset(df_raw, window="60s")
os.makedirs(os.path.dirname(output_features_path), exist_ok=True)
df_features.to_csv(output_features_path, index=False)

print(f"Successfully processed {len(df_features)} records into feature matrix.")
print(f"Saved feature dataset to: {output_features_path}")
df_features.head()

In [ ]:
# Feature Summary & Correlation with Target
numeric_features = [c for c in df_features.select_dtypes(include=[np.number]).columns if c not in ["label"]]
print(f"Total numeric features generated: {len(numeric_features)}")
print(numeric_features)

if "label" in df_features.columns:
    corr = df_features[numeric_features].apply(lambda x: x.corr(df_features["label"])).sort_values(ascending=False)
    print("\nFeature correlation with malicious label:")
    print(corr.round(4).to_string())